# 6 — Chatbot (RAG + Gemini 2.5 Flash)

Full RAG chatbot for taxes.gov.az Q&A.

| | |
|---|---|
| **Retrieval** | ChromaDB + multilingual-e5-large |
| **Generation** | Gemini 2.5 Flash (free tier) |
| **Language** | Azerbaijani |
| **DB** | `../data/db/chroma/` |
| **Logs** | `../data/logs/query_log.jsonl` |

### Pipeline
```
User query → embed (query: prefix) → ChromaDB search → distance filter
           → build context → Gemini 2.5 Flash → Azerbaijani answer
```

## 0 — Install

In [ ]:
# Run once — comment out after first install
# !pip install google-genai sentence-transformers chromadb

## 1 — Imports & config

In [ ]:
import json
import logging
import os
import time
from datetime import datetime, timezone
from pathlib import Path

import chromadb
from google import genai
from google.genai import errors, types
from sentence_transformers import SentenceTransformer

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

In [9]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
# ── Config ────────────────────────────────────────────────────────────────────
DB_PATH         = Path("../data/db/chroma")
COLLECTION_NAME = "taxes_az_qa"
LOG_PATH        = Path("../data/logs/query_log.jsonl")

EMBED_MODEL     = "intfloat/multilingual-e5-large"
E5_QUERY_PREFIX = "query: "   # passage: was used at index time — DO NOT change

GEMINI_MODEL      = "gemini-2.5-flash"
TOP_K             = 5
DISTANCE_THRESHOLD = 0.5    # cosine distance — drop above this
MAX_OUTPUT_TOKENS = 1024
TEMPERATURE       = 0.2     # low = factual — correct for tax Q&A
THINKING_BUDGET   = 0       # 0 = fast mode, no chain-of-thought

SYSTEM_PROMPT = """Siz Azərbaycan Respublikasının Dövlət Vergi Xidmətinin rəsmi \
saytı olan taxes.gov.az-ın sual-cavab məlumat bazasına əsaslanan köməkçi \
chatbotsunuz.

Qaydalar:
1. Yalnız Azərbaycan dilində cavab verin.
2. Cavabınızı YALNIZ verilmiş kontekst məlumatlarına əsaslandırın.
3. Kontekstdə cavab yoxdursa — "Bu sual barədə məlumat bazamda məlumat tapılmadı." \
deyin. Heç vaxt uydurmayın.
4. Mənbə nömrəsinə istinad edin (məsələn: [Mənbə 1]).
5. Vergi Məcəlləsinə istinadlar varsa, onları cavabda qoruyun.
6. Qısa və aydın cavab verin."""

print("Config loaded")

Config loaded


## 2 — Initialise resources

In [4]:
# Embedding model
log.info(f"Loading {EMBED_MODEL}...")
embed_model = SentenceTransformer(EMBED_MODEL)
print(f"Embedding model ready — dim: {embed_model.get_sentence_embedding_dimension()}")

16:56:16 | INFO | Loading intfloat/multilingual-e5-large...
16:56:16 | INFO | No device provided, using cuda:0
16:56:16 | INFO | HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-large/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
16:56:16 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/intfloat/multilingual-e5-large/3d7cfbdacd47fdda877c5cd8a79fbcc4f2a574f3/modules.json "HTTP/1.1 200 OK"
16:56:16 | INFO | HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-large/resolve/main/config_sentence_transformers.json "HTTP/1.1 404 Not Found"
16:56:16 | INFO | Loading SentenceTransformer model from intfloat/multilingual-e5-large.
16:56:17 | INFO | HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-large/resolve/main/config_sentence_transformers.json "HTTP/1.1 404 Not Found"
16:56:17 | INFO | HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-large/resolve/main/README.md "HTTP/1.1 307 Tempora

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

16:56:19 | INFO | HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-large/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
16:56:19 | INFO | HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-large/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
16:56:19 | INFO | HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-large/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
16:56:20 | INFO | HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-large/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
16:56:20 | INFO | HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-large/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
16:56:20 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/intfloat/multilingual-e5-large/3d7cfbdacd47fdda877c5cd8a79fbcc4f2a574f3/tokenizer_config.json "HTTP/1.1 200 OK"
16:56:20 | INFO | HTTP R

Embedding model ready — dim: 1024


C:\Users\elihu\AppData\Local\Temp\ipykernel_20436\1817417920.py:4: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Embedding model ready — dim: {embed_model.get_sentence_embedding_dimension()}")


In [5]:
# ChromaDB
db_client  = chromadb.PersistentClient(path=str(DB_PATH))
collection = db_client.get_collection(name=COLLECTION_NAME)
print(f"Collection '{COLLECTION_NAME}' — {collection.count():,} docs")

Collection 'taxes_az_qa' — 3,558 docs


In [10]:
# Gemini client
# Set your key: os.environ['GEMINI_API_KEY'] = 'your_key'  ← run this first if needed
api_key = os.environ.get("GEMINI_API_KEY")
if not api_key:
    raise EnvironmentError(
        "GEMINI_API_KEY not set.\n"
        "Run: os.environ['GEMINI_API_KEY'] = 'your_key'"
    )

gemini_client = genai.Client(api_key=api_key)
print(f"Gemini client ready — model: {GEMINI_MODEL}")

Gemini client ready — model: gemini-2.5-flash


## 3 — Pipeline functions

In [11]:
def embed_query(query: str) -> list[float]:
    """Embed with E5 query prefix — must match passage: used at index time."""
    vec = embed_model.encode(
        [E5_QUERY_PREFIX + query],
        normalize_embeddings=True,
        convert_to_numpy=True,
    )
    return vec[0].tolist()


def search(query: str, top_k: int = TOP_K, threshold: float = DISTANCE_THRESHOLD) -> list[dict]:
    """Embed query, search ChromaDB, filter by distance threshold."""
    query_vec = embed_query(query)

    raw = collection.query(
        query_embeddings=[query_vec],
        n_results=top_k,
        include=["metadatas", "distances", "documents"],
    )

    results = []
    for i in range(len(raw["ids"][0])):
        distance = raw["distances"][0][i]
        if distance > threshold:
            continue
        meta = raw["metadatas"][0][i]
        results.append({
            "rank":        i + 1,
            "id":          raw["ids"][0][i],
            "distance":    round(distance, 4),
            "similarity":  round(1 - distance / 2, 4),
            "source_id":   meta["source_id"],
            "chunk_type":  meta["chunk_type"],
            "answer_date": meta["answer_date"],
            "read_count":  meta["read_count"],
            "question":    meta["question"],
            "answer":      meta["answer"],
            "source_url":  meta["source_url"],
        })

    return results


def build_context(results: list[dict]) -> str:
    """Format retrieved chunks into numbered context block."""
    if not results:
        return ""
    parts = []
    for r in results:
        parts.append(
            f"[Mənbə {r['rank']}] (uyğunluq: {r['similarity']:.0%})\n"
            f"Sual: {r['question'].strip()}\n"
            f"Cavab: {r['answer'].strip()}"
        )
    return "\n\n".join(parts)


def build_messages(user_query: str, context: str) -> list[types.Content]:
    """Build Gemini message list with context injected into user turn."""
    if context:
        user_text = f"Kontekst məlumatları:\n{context}\n\nSualım: {user_query}"
    else:
        user_text = (
            f"Kontekst məlumatları: Heç bir uyğun nəticə tapılmadı.\n\n"
            f"Sualım: {user_query}"
        )
    return [
        types.Content(role="user", parts=[types.Part(text=user_text)])
    ]


def call_llm(messages: list[types.Content], retries: int = 3) -> str:
    """Call Gemini 2.5 Flash with retry on rate limits and server errors."""
    config = types.GenerateContentConfig(
        system_instruction=SYSTEM_PROMPT,
        temperature=TEMPERATURE,
        max_output_tokens=MAX_OUTPUT_TOKENS,
        thinking_config=types.ThinkingConfig(thinking_budget=THINKING_BUDGET),
    )
    for attempt in range(1, retries + 1):
        try:
            response = gemini_client.models.generate_content(
                model=GEMINI_MODEL,
                contents=messages,
                config=config,
            )
            return response.text
        except errors.ClientError as e:
            if "429" in str(e) or "quota" in str(e).lower():
                wait = 2 ** attempt
                log.warning(f"Rate limit (attempt {attempt}/{retries}), waiting {wait}s...")
                time.sleep(wait)
            else:
                raise
        except errors.ServerError as e:
            wait = 2 ** attempt
            log.warning(f"Server error (attempt {attempt}/{retries}): {e}, waiting {wait}s...")
            time.sleep(wait)
    raise RuntimeError(f"LLM call failed after {retries} retries.")


def log_query(query, results, answer_text, retrieval_ms, llm_ms):
    """Append query + result metadata to JSONL log file."""
    LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
    entry = {
        "ts":             datetime.now(timezone.utc).isoformat(),
        "query":          query,
        "n_results":      len(results),
        "top_distances":  [r["distance"] for r in results[:3]],
        "source_ids":     [r["source_id"] for r in results],
        "answer_preview": answer_text[:200],
        "retrieval_ms":   round(retrieval_ms),
        "llm_ms":         round(llm_ms),
        "model":          GEMINI_MODEL,
    }
    with open(LOG_PATH, "a", encoding="utf-8") as f:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")


def answer(query: str) -> dict:
    """Full RAG pipeline. Returns answer + sources + timing."""
    t0      = time.time()
    results = search(query)
    retrieval_ms = (time.time() - t0) * 1000

    context  = build_context(results)
    messages = build_messages(query, context)

    t1 = time.time()
    llm_answer = call_llm(messages)
    llm_ms = (time.time() - t1) * 1000

    log_query(query, results, llm_answer, retrieval_ms, llm_ms)

    return {
        "query":         query,
        "answer":        llm_answer,
        "sources":       results,
        "retrieval_ms":  retrieval_ms,
        "llm_ms":        llm_ms,
    }


print("All functions defined")

All functions defined


## 4 — Test single query

In [12]:
# Test retrieval only first (no LLM cost)
test_query = "VÖEN-in ləğvi proseduru necədir?"

results = search(test_query)
print(f"Query: {test_query}")
print(f"Results: {len(results)} passed threshold")
print()
for r in results:
    print(f"  Rank {r['rank']} | similarity: {r['similarity']:.0%} | distance: {r['distance']}")
    print(f"  Q: {r['question'][:100]}")
    print(f"  A: {r['answer'][:100]}")
    print()

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Query: VÖEN-in ləğvi proseduru necədir?
Results: 5 passed threshold

  Rank 1 | similarity: 94% | distance: 0.1278
  Q: Salam. VÖEN-mi necə ləğv edə bilərəm?
  A: Bildiririk ki, vergi uçotundan çıxmaq üçün vergi ödəyicisi və ya onun səlahiyyətli nümayəndəsi (nota

  Rank 2 | similarity: 93% | distance: 0.1316
  Q: VÖEN-in ləğv edilməsi.
  A: Bildiririk ki, vergi uçotunuzun ləğv edilməsi üçün “Fiziki şəxsin vergi uçotundan çıxarılması haqqın

  Rank 3 | similarity: 93% | distance: 0.132
  Q: 2004237452 nömrəli VÖEN-in ləğv olunması ücün uzun muddətdir ki, muraciət etmişəm. Ancaq hələ ləğv o
  A: Bildiririk ki, fiziki şəxs vergi uçotunun ləğv edilməsi haqda müraciət etdikdə, vergi orqanında müva

  Rank 4 | similarity: 93% | distance: 0.1321
  Q: Salam, mən 2014-cü ildə VÖEN açdırmışam, heç bir sahibkarlıqla məşğul  olmamışam və ehtiyac olmadığı
  A: Bildiririk ki, vergi uçotunuzun ləğv edilməsi üçün “Fiziki şəxsin vergi uçotundan çıxarılması haqqın

  Rank 5 | similarity: 93% | distance

In [13]:
# Inspect context that will be sent to LLM
context = build_context(results)
print("Context sent to LLM:")
print("─" * 60)
print(context[:1000], "..." if len(context) > 1000 else "")
print("─" * 60)
print(f"Total context length: {len(context)} chars, ~{len(context)//4} tokens")

Context sent to LLM:
────────────────────────────────────────────────────────────
[Mənbə 1] (uyğunluq: 94%)
Sual: Salam. VÖEN-mi necə ləğv edə bilərəm?
Cavab: Bildiririk ki, vergi uçotundan çıxmaq üçün vergi ödəyicisi və ya onun səlahiyyətli nümayəndəsi (notarial qaydada təsdiq edilmiş etibarnamə ilə) tərəfindən uçotda olduğu vergi orqanına və ya onun əhatə dairəsinə aid olan vergi ödəyicilərinə xidmət mərkəzlərinə gəlməklə və ya poçt vasitəsilə “Fiziki şəxsin vergi uçotundan çıxarılması haqqında ərizə” və vergi şəhadətnaməsi vergi orqanından təhvil götürüldüyü halda onun əsli təqdim edilməlidir. Vergi şəhadətnaməsi itirildikdə isə vergi ödəyicisi tərəfindən yazılı məlumat əlavə olaraq təqdim edilməlidir.
Onu da qeyd edək ki, gücləndirilmiş elektron imza (o cümlədən Asan İmza) olduqda Dövlət Vergi Xidmətinin İnternet Vergi İdarəsi portalı üzərindən fiziki şəxsin fəaliyyətinin onlayn rejimdə ləğvi mümkündür.

[Mənbə 2] (uyğunluq: 93%)
Sual: VÖEN-in ləğv edilməsi.
Cavab: Bildiririk ki, v

In [14]:
# Full pipeline — calls Gemini API (uses free quota)
result = answer(test_query)

print(f"Query: {result['query']}")
print()
print("Answer:")
print("─" * 60)
print(result["answer"])
print("─" * 60)
print(f"Sources used: {len(result['sources'])}")
print(f"Retrieval: {result['retrieval_ms']:.0f}ms | LLM: {result['llm_ms']:.0f}ms")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

17:00:53 | INFO | AFC is enabled with max remote calls: 10.
17:01:02 | INFO | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"


Query: VÖEN-in ləğvi proseduru necədir?

Answer:
────────────────────────────────────────────────────────────
VÖEN-in ləğvi üçün "Fiziki şəxsin vergi uçotundan çıxarılması haqqında ərizə" və uçota alınmanı təsdiq edən şəhadətnamə (vergi orqanından təhvil götürüldüyü halda onun əsli) təqdim edilməlidir. Bu sənədləri şəxsən, səlahiyyətli nümayəndə (notarial qaydada təsdiq edilmiş etibarnamə ilə) vasitəsilə uçotda olduğunuz vergi orqanına və ya onun əhatə dairəsinə aid olan vergi ödəyicilərinə xidmət mərkəzlərinə gəlməklə, yaxud poçt vasitəsilə təqdim edə bilərsiniz. [Mənbə 1, Mənbə 2, Mənbə 4, Mənbə 5]

Əgər gücləndirilmiş elektron imzanız (o cümlədən Asan İmza) varsa, Dövlət Vergi Xidmətinin İnternet Vergi İdarəsi portalı üzərindən fiziki şəxsin fəaliyyətinin onlayn rejimdə ləğvi mümkündür. [Mənbə 1, Mənbə 2, Mənbə 4, Mənbə 5]

Vergi şəhadətnaməsi itirildikdə, vergi ödəyicisi tərəfindən yazılı məlumat əlavə olaraq təqdim edilməlidir. [Mənbə 1]

Fiziki şəxs vergi uçotunun ləğv edilməsi h

## 5 — Test multiple queries

In [15]:
test_queries = [
    "Gəlir vergisi neçə faizdir?",
    "ƏDV qeydiyyatı üçün dövriyyə həddi nə qədərdir?",
    "Kriptovalyuta vergisi necə hesablanır?",
    "Bu sual heç bir mənbədə yoxdur — test üçün",  # should return 'not found'
]

for q in test_queries:
    print(f"\nQuery: {q}")
    r = answer(q)
    print(f"Answer: {r['answer'][:300]}")
    print(f"Sources: {len(r['sources'])} | {r['retrieval_ms']:.0f}ms + {r['llm_ms']:.0f}ms")
    print("─" * 60)
    time.sleep(6)  # respect 10 RPM free tier limit


Query: Gəlir vergisi neçə faizdir?


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

17:01:09 | INFO | AFC is enabled with max remote calls: 10.
17:01:12 | INFO | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"


Answer: Gəlir vergisi fəaliyyət növündən asılı olaraq dəyişir:

*   **Kriptovalyuta satışından əldə edilən gəlir:** 14 faiz dərəcə ilə gəlir vergisi hesablanır [Mənbə 1].
*   **Əmlakın icarəyə verilməsindən əldə olunan gəlir:** 14 faiz dərəcə ilə vergi tutulur [Mənbə 2].
*   **Səhm alğı-satqısından əldə edi
Sources: 5 | 316ms + 2742ms
────────────────────────────────────────────────────────────

Query: ƏDV qeydiyyatı üçün dövriyyə həddi nə qədərdir?


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

17:01:28 | INFO | AFC is enabled with max remote calls: 10.
17:01:29 | INFO | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 503 Service Unavailable"
17:01:29 | WARNING | Server error (attempt 1/3): 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}, waiting 2s...
17:01:31 | INFO | AFC is enabled with max remote calls: 10.
17:01:34 | INFO | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"


Answer: Kriptovalyuta vergisi, Vergi Məcəlləsinin 99.3.8-ci maddəsinə əsasən, vergi ödəyicisinin aktivlərinin (o cümlədən kriptovalyutanın) ilkin qiymətinin artdığını göstərən hər hansı digər gəlir (təqdim olunduğu təqdirdə) - əmək haqqından başqa onun qeyri-sahibkarlıq fəaliyyətindən əldə olunan gəliri hes
Sources: 5 | 302ms + 6465ms
────────────────────────────────────────────────────────────

Query: Bu sual heç bir mənbədə yoxdur — test üçün


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

17:01:40 | INFO | AFC is enabled with max remote calls: 10.
17:01:41 | INFO | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"


Answer: Bu sual barədə məlumat bazamda məlumat tapılmadı.
Sources: 5 | 370ms + 1054ms
────────────────────────────────────────────────────────────


## 6 — Inspect query log

In [16]:
# Read and display all logged queries
if LOG_PATH.exists():
    with open(LOG_PATH, encoding="utf-8") as f:
        logs = [json.loads(line) for line in f if line.strip()]

    print(f"Total logged queries: {len(logs)}")
    print()
    for entry in logs[-5:]:  # show last 5
        print(f"  [{entry['ts'][:19]}] {entry['query'][:60]}")
        print(f"    results={entry['n_results']} | "
              f"distances={entry['top_distances']} | "
              f"retrieval={entry['retrieval_ms']}ms | "
              f"llm={entry['llm_ms']}ms")
        print()
else:
    print("No log file yet")

Total logged queries: 5

  [2026-05-11T13:01:02] VÖEN-in ləğvi proseduru necədir?
    results=5 | distances=[0.1278, 0.1316, 0.132] | retrieval=220ms | llm=9036ms

  [2026-05-11T13:01:12] Gəlir vergisi neçə faizdir?
    results=5 | distances=[0.1353, 0.1359, 0.1368] | retrieval=316ms | llm=2742ms

  [2026-05-11T13:01:21] ƏDV qeydiyyatı üçün dövriyyə həddi nə qədərdir?
    results=5 | distances=[0.1083, 0.1328, 0.1328] | retrieval=279ms | llm=2768ms

  [2026-05-11T13:01:34] Kriptovalyuta vergisi necə hesablanır?
    results=5 | distances=[0.0942, 0.106, 0.1212] | retrieval=302ms | llm=6465ms

  [2026-05-11T13:01:41] Bu sual heç bir mənbədə yoxdur — test üçün
    results=5 | distances=[0.1377, 0.1432, 0.1452] | retrieval=370ms | llm=1054ms



## 7 — Tune distance threshold

In [17]:
# Check what distances look like across your queries
# Use this to decide if DISTANCE_THRESHOLD needs adjusting
# Rule of thumb:
#   < 0.25 = very strong match
#   0.25 - 0.40 = good match
#   0.40 - 0.50 = weak match — borderline
#   > 0.50 = likely irrelevant — correct to filter out

if LOG_PATH.exists():
    with open(LOG_PATH, encoding="utf-8") as f:
        logs = [json.loads(line) for line in f if line.strip()]

    all_distances = [d for entry in logs for d in entry["top_distances"]]
    if all_distances:
        print(f"Distance stats across {len(all_distances)} top results:")
        print(f"  min  : {min(all_distances):.4f}")
        print(f"  max  : {max(all_distances):.4f}")
        print(f"  mean : {sum(all_distances)/len(all_distances):.4f}")
        print()
        print(f"Current threshold : {DISTANCE_THRESHOLD}")
        filtered = sum(1 for d in all_distances if d > DISTANCE_THRESHOLD)
        print(f"Would filter out  : {filtered}/{len(all_distances)} results ({filtered/len(all_distances):.0%})")

Distance stats across 15 top results:
  min  : 0.0942
  max  : 0.1452
  mean : 0.1281

Current threshold : 0.5
Would filter out  : 0/15 results (0%)
